# Stochastic Calculus, Step by Step · Lesson 30: Choosing a method

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Where a lesson needs the share's real-world expected return, it is μ = 8%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. One option, every method · 3. Early exercise · 4. Many assets · 5. Payoffs that depend on the path · 6. Greeks from each method · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 30,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
mu = 0.08                         # the share's real-world expected return
theta_mpr = (mu - r) / sigma      # the market price of risk
Nv = np.vectorize(N)              # N, for arrays
phi = lambda x: np.exp(-np.asarray(x, float) ** 2 / 2) / math.sqrt(2 * math.pi)   # the normal density


def d_plus(s, tau, sig=sigma, k=K):
    return (np.log(s / k) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))


def d_minus(s, tau, sig=sigma, k=K):
    return d_plus(s, tau, sig, k) - sig * np.sqrt(tau)


def call(s, tau, sig=sigma, k=K):
    """The Black-Scholes call price: Lesson 20 derives it."""
    return s * Nv(d_plus(s, tau, sig, k)) - k * np.exp(-r * tau) * Nv(d_minus(s, tau, sig, k))


def put(s, tau, sig=sigma, k=K):
    return call(s, tau, sig, k) - s + k * np.exp(-r * tau)


out["bs"] = {"mu": mu, "theta": theta_mpr, "d_plus": float(d_plus(S0, T)), "d_minus": float(d_minus(S0, T)),
             "N_plus": N(float(d_plus(S0, T))), "N_minus": N(float(d_minus(S0, T))),
             "call": float(call(S0, T)), "put": float(put(S0, T))}
out["bs"]

## 2. One option, every method

Feynman-Kac (Lesson 28) has two sides. The **PDE side** works backwards on a grid: trees and finite differences. The **expectation side** works forwards on paths: Monte Carlo. All of them price the same call. To compare fairly, count the work as multiplications or draws, not seconds, so that the numbers are the same on any computer.

In [ ]:
bs_c = float(call(S0, T))
disc = math.exp(-r * T)


def binomial(n_, kind="call", american=False, s0=S0):
    dt = T / n_
    u_ = math.exp(sigma * math.sqrt(dt)); d_ = 1 / u_; p_ = (math.exp(r * dt) - d_) / (u_ - d_); df = math.exp(-r * dt)
    pay = (lambda s: np.maximum(s - K, 0)) if kind == "call" else (lambda s: np.maximum(K - s, 0))
    v = pay(s0 * u_ ** (2 * np.arange(n_ + 1) - n_))
    for i in range(n_ - 1, -1, -1):
        v = df * (p_ * v[1:] + (1 - p_) * v[:-1])
        if american:
            v = np.maximum(v, pay(s0 * u_ ** (2 * np.arange(i + 1) - i)))
    return float(v[0])


def trinomial(n_):
    dt = T / n_; nu = r - 0.5 * sigma**2
    gap = math.log(K / S0)
    j = max(1, round(gap / (math.sqrt(3) * sigma * math.sqrt(dt))))
    dx = gap / j
    pu_ = 0.5 * (sigma**2 * dt / dx**2 + nu * dt / dx); pd_ = 0.5 * (sigma**2 * dt / dx**2 - nu * dt / dx); pm_ = 1 - pu_ - pd_
    v = np.maximum(S0 * np.exp(np.arange(-n_, n_ + 1) * dx) - K, 0)
    for _ in range(n_):
        v = math.exp(-r * dt) * (pu_ * v[2:] + pm_ * v[1:-1] + pd_ * v[:-2])
    return float(v[0])


def explicit_fd(M_, N_):
    nu = r - 0.5 * sigma**2
    half = 5 * sigma * math.sqrt(T)
    y = math.log(S0) + np.linspace(-half, half, M_ + 1); dy = y[1] - y[0]; dtau = T / N_
    a_ = dtau * sigma**2 / (2 * dy**2); b_ = dtau * nu / (2 * dy)
    s_ = np.exp(y); v = np.maximum(s_ - K, 0)
    for n in range(1, N_ + 1):
        new = v.copy()
        new[1:-1] = ((a_ + b_) * v[2:] + (1 - 2 * a_) * v[1:-1] + (a_ - b_) * v[:-2]) / (1 + r * dtau)
        new[0], new[-1] = 0.0, s_[-1] - K * math.exp(-r * n * dtau)
        v = new
    return float(np.interp(math.log(S0), y, v))


rng = np.random.default_rng(seed)
Z_mc = rng.standard_normal(1_000_000)
pay_mc = disc * np.maximum(S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * Z_mc) - K, 0)
all_rows = [{"method": "Formula", "side": "both", "value": bs_c, "error": 0.0, "work": 1},
            {"method": "Binomial tree, 500 steps", "side": "PDE", "value": binomial(500), "error": binomial(500) - bs_c, "work": 500 * 501 // 2},
            {"method": "Trinomial, strike on a node, 450 steps", "side": "PDE", "value": trinomial(450), "error": trinomial(450) - bs_c, "work": 451 * 451},
            {"method": "Explicit differences, 200 x 800", "side": "PDE", "value": explicit_fd(200, 800), "error": explicit_fd(200, 800) - bs_c, "work": 200 * 800},
            {"method": "Monte Carlo, 1,000,000 paths", "side": "paths", "value": float(pay_mc.mean()), "error": float(pay_mc.mean() - bs_c), "work": 1_000_000}]
out["all"] = {"rows": all_rows, "mc_se": float(pay_mc.std() / 1000)}
pd.DataFrame(all_rows)

## 3. Early exercise

At each moment, an American put asks whether exercising beats continuing. On a tree or grid, the continuation value is right there at each node. On a path, it isn't: you can't see what lies ahead without looking into the future. **Longstaff-Schwartz** estimates it by regressing the later discounted cash flows on the current share price.

In [ ]:
def lsm(paths, n_ex, gen, degree=3):
    # Longstaff-Schwartz for the American put, with polynomial regression on S/K. The same paths fit and value it.
    dt = T / n_ex
    dW_ = gen.standard_normal((paths, n_ex))
    S_ = S0 * np.exp(((r - 0.5 * sigma**2) * dt + sigma * math.sqrt(dt) * dW_).cumsum(axis=1))
    cash = np.maximum(K - S_[:, -1], 0.0)
    df = math.exp(-r * dt)
    for i in range(n_ex - 2, -1, -1):
        cash *= df
        itm = K - S_[:, i] > 0
        if itm.sum() > degree + 1:
            x = S_[itm, i] / K
            coef = np.polyfit(x, cash[itm], degree)
            cont = np.polyval(coef, x)
            ex_now = K - S_[itm, i]
            do = ex_now > cont
            idx = np.where(itm)[0][do]
            cash[idx] = ex_now[do]
    cash *= df
    return float(cash.mean()), float(cash.std(ddof=1) / math.sqrt(paths))


gen = np.random.default_rng(seed + 1)
tree_am = 0.5 * (binomial(4000, "put", True) + binomial(4001, "put", True))
bs_p = float(put(S0, T))
lsm_rows = []
for paths, ex_n in ((20_000, 12), (200_000, 12), (200_000, 50), (200_000, 252)):
    v_, se_ = lsm(paths, ex_n, gen)
    lsm_rows.append({"paths": paths, "dates": ex_n, "price": v_, "se": se_, "gap": v_ - tree_am})
out["am"] = {"rows": lsm_rows, "tree": tree_am, "european": bs_p, "premium": tree_am - bs_p}
pd.DataFrame(lsm_rows)

## 4. Many assets

A grid needs a point for every combination of share prices: $M^d$ points for $d$ shares. Monte Carlo's error is $s/\sqrt{\text{paths}}$ whatever $d$ is. Beyond about three assets, the grid has no chance. A five-share basket call:

In [ ]:
grid_rows = [{"d": d, "points": 100.0**d, "bytes": 8 * 100.0**d, "mc_draws": d * 1_000_000} for d in (1, 2, 3, 4, 5, 10)]
d_ = 5; rho_ = 0.5
L = np.linalg.cholesky(np.full((d_, d_), rho_) + (1 - rho_) * np.eye(d_))
gen = np.random.default_rng(seed + 2)
Zb = gen.standard_normal((1_000_000, d_)) @ L.T
S_b = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * Zb)
pay_b = disc * np.maximum(S_b.mean(axis=1) - K, 0)
# a lognormal match, as a check on the Monte Carlo
F_ = S0 * math.exp(r * T)
m2 = (F_**2 / d_**2) * sum(math.exp((sigma**2 if i == j else rho_ * sigma**2) * T) for i in range(d_) for j in range(d_))
vol_b = math.sqrt(math.log(m2 / F_**2) / T)
out["many"] = {"rows": grid_rows, "rho": rho_, "d": d_, "basket": float(pay_b.mean()), "se": float(pay_b.std() / 1000), "vol_match": vol_b,
               "approx": float(call(S0, T, vol_b)), "single": bs_c}
pd.DataFrame(grid_rows)

## 5. Payoffs that depend on the path

An **Asian** call pays on the average price along the way, an average that depends on the whole path. On paths, that's trivial. A grid must add the running average as a new dimension. With 12 monthly dates and 252 daily ones, compared with the European call:

In [ ]:
gen = np.random.default_rng(seed + 3)
asian_rows = []
for dates in (12, 52, 252):
    dt = T / dates
    log_inc = (r - 0.5 * sigma**2) * dt + sigma * math.sqrt(dt) * gen.standard_normal((200_000, dates))
    S_p = S0 * np.exp(log_inc.cumsum(axis=1))
    pay_a = disc * np.maximum(S_p.mean(axis=1) - K, 0)
    asian_rows.append({"dates": dates, "price": float(pay_a.mean()), "se": float(pay_a.std() / math.sqrt(200_000))})
out["asian"] = {"rows": asian_rows, "european": bs_c, "ratio": asian_rows[-1]["price"] / bs_c}
pd.DataFrame(asian_rows)

## 6. Greeks from each method

Grids give the Greeks almost free: delta and gamma come from neighbouring nodes. Paths need more thought. **Bumping** uses a small change in $S_0$ with the *same* random numbers. The **pathwise** method differentiates the payoff along each path. The **likelihood ratio** method differentiates the density instead, and works for digitals, whose payoff has no useful derivative.

In [ ]:
gen = np.random.default_rng(seed + 4)
M_g = 1_000_000
Zg = gen.standard_normal(M_g)
Zg2 = gen.standard_normal(M_g)
drift_ = (r - 0.5 * sigma**2) * T; vol_ = sigma * math.sqrt(T)
ST = S0 * np.exp(drift_ + vol_ * Zg)
pay = lambda s, z=Zg: disc * np.maximum(s * np.exp(drift_ + vol_ * z) - K, 0)
bump = 1.0
same = (pay(S0 + bump) - pay(S0 - bump)) / (2 * bump)
diff = (pay(S0 + bump) - pay(S0 - bump, Zg2)) / (2 * bump)
pathwise = disc * (ST > K) * ST / S0
lr = disc * np.maximum(ST - K, 0) * Zg / (S0 * sigma * math.sqrt(T))
true_delta = float(Nv(d_plus(S0, T)))
se = lambda x: float(x.std() / math.sqrt(M_g))
g_rows = [{"method": "Bump, different random numbers", "value": float(diff.mean()), "se": se(diff)},
          {"method": "Bump, same random numbers", "value": float(same.mean()), "se": se(same)},
          {"method": "Pathwise", "value": float(pathwise.mean()), "se": se(pathwise)},
          {"method": "Likelihood ratio", "value": float(lr.mean()), "se": se(lr)}]
dig_true = disc * float(phi(d_minus(S0, T))) / (S0 * sigma * math.sqrt(T))
dig_lr = disc * (ST > K) * Zg / (S0 * sigma * math.sqrt(T))
dig_rows = []
for b_ in (5.0, 1.0, 0.1, 0.01):
    dd_ = disc * ((S0 + b_) * np.exp(drift_ + vol_ * Zg) > K).astype(float) - disc * ((S0 - b_) * np.exp(drift_ + vol_ * Zg) > K).astype(float)
    dd_ = dd_ / (2 * b_)
    dig_rows.append({"bump": b_, "value": float(dd_.mean()), "se": se(dd_)})
h_grid = 1.0
n_gr = 1000
tree_delta = (binomial(n_gr, s0=S0 + h_grid) - binomial(n_gr, s0=S0 - h_grid)) / (2 * h_grid)
out["greeks"] = {"rows": g_rows, "true": true_delta, "tree_delta": tree_delta, "digital_true": dig_true, "digital_lr": float(dig_lr.mean()), "digital_lr_se": se(dig_lr),
                 "digital_rows": dig_rows, "digital_pathwise": 0.0}
pd.DataFrame(g_rows)

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 4: Solutions*.

**Exercise 1.** A one-share tree has $n$ steps with cost $\propto n^2$ and error $\propto 1/n$. A Monte Carlo price has $M$ paths with cost $\propto M$ and error $\propto 1/\sqrt M$. How does the cost of each grow as the target error $\varepsilon$ falls? Which wins at small $\varepsilon$, and what changes with more assets?

**Exercise 2.** A grid has 100 points per asset. How many points, and how much memory at 8 bytes each, for $d = 1, 2, 3, 5, 10$? How many time steps would an explicit scheme need, given the stability limit? Roughly when is a grid impossible on a laptop?

**Exercise 3.** For each of these, name a suitable method and say why: a European put; an American put on one share; a call on the average of 20 shares; a one-share barrier option that dies if the price touches 80; an American option on 5 shares; the delta of a digital on one share.

**Exercise 4.** Derive the pathwise delta: differentiate $e^{-rT}\max(S_T - K, 0)$ along the path, with $S_T = S_0e^{(r - \frac12\sigma^2)T + \sigma\sqrt TZ}$ and $Z$ fixed. Show that its expectation is $N(d_+)$. Why is it fine that the payoff has a kink?

**Exercise 5.** Why does the pathwise method fail for a digital? Derive the likelihood-ratio formula: write the price as $\int f(s)\,p(s; S_0)\,ds$, differentiate the density, and show the weight is $Z/(S_0\sigma\sqrt T)$.

**Exercise 6.** In Longstaff-Schwartz, the exercise rule is fitted to the paths, and then the same paths value the option. Explain why, if a fresh set of paths is used for valuation, the answer must be a low estimate of the true American value. Why can reusing the same paths push it up?

**Exercise 7.** (Code) Run Longstaff-Schwartz with polynomial degrees 1, 2, 3, 5 and exercise dates 12, 50. Use one set of paths to fit the rule and a separate set to value it. Compare with the tree. How much does each choice matter?

**Exercise 8.** (Code) Compute the delta of the cash digital by bumping with same random numbers at bumps 5, 1, 0.1, 0.01, and by the likelihood-ratio formula, and compare with the exact $e^{-rT}\varphi(d_-)/(S_0\sigma\sqrt T)$. Why does the bump's standard error rise as the bump shrinks? Which would you use?

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")